# Cassini Dust Analyzer: QI-Amplitude Ablation Study ($k=1$)

This notebook conducts a specialized ablation study focusing on the impact of **QI-amplitude (impact charge / peak amplitude)** of the few-shot reference spectra on the VLM's classification accuracy.

### Study Objective
When doing $k=1$ few-shot classification, does the quality or intensity (QI-amplitude) of the reference spectrum affect accuracy? 
To test this:
1. We take the non-noise training spectra and calculate their $\log_{10}(\text{qi\_ampl})$ values (applying $\log_{10}$ to handle the high dynamic range).
2. We split this range into 5 equal-width bins (from Bin 0 for the lowest amplitude to Bin 4 for the highest).
3. For each bin, we perform **3 independent runs** to get statistical variance. In each run, we randomly select a training spectrum from that bin's range as the $k=1$ few-shot reference for each class (except for `Noise`, which is sampled randomly from the Noise pool without filtering).
4. We evaluate the VLM using up to 100 test samples per class (excluding training samples) to get accuracy metrics.

### Folder Structure and Output
All intermediate files, predictions, parquet results, confusion matrices, and the final accuracy curve are stored in `qi_ablation_results/`.

In [1]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
from dotenv import load_dotenv

# Ensure we can import from the workspace root
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from cda_dust_agent.config import Config

load_dotenv(dotenv_path="../.env")
configs = Config()
print(f"Using VLM Model: {configs.agent_settings.model}")

Using VLM Model: gemini-3.5-flash


In [2]:
# Inspect the training data and check the QI Amplitude distribution
from scripts.run_ablation_study_incremental import preprocess_huggingface_data, get_or_create_train_sclks

full_df = preprocess_huggingface_data("H")
train_sclks = get_or_create_train_sclks(full_df)

# Extract SCLK lists
all_train_sclks = []
for cls, sclks in train_sclks.items():
    all_train_sclks.extend(sclks)

train_df = full_df[full_df['sclk'].isin(all_train_sclks)].copy()
train_df['log_qi'] = np.log10(train_df['qi_ampl'])

non_noise_train = train_df[train_df['class'] != 'Noise']
min_log_qi = non_noise_train['log_qi'].min()
max_log_qi = non_noise_train['log_qi'].max()

print(f"Training set non-noise QI amplitude range (log10): {min_log_qi:.4f} to {max_log_qi:.4f}")

# Display decile bin boundaries
bin_edges = np.linspace(min_log_qi, max_log_qi, 6)
print("\nBin boundaries for 5 equal-width log10(QI) bins:")
for i in range(5):
    print(f"  Bin {i}: [{bin_edges[i]:.4f}, {bin_edges[i+1]:.4f})")

Loading existing training SCLKs from cda_dust_agent/data/ablation_study/train_sclks.json...
Training set non-noise QI amplitude range (log10): -13.9924 to -12.1279

Bin boundaries for 5 equal-width log10(QI) bins:
  Bin 0: [-13.9924, -13.6195)
  Bin 1: [-13.6195, -13.2466)
  Bin 2: [-13.2466, -12.8737)
  Bin 3: [-12.8737, -12.5008)
  Bin 4: [-12.5008, -12.1279)


In [3]:
# Execute the full QI Ablation Study runner script
# This generates requests, runs the Vertex AI Batch prediction, parses, and plots results.
from scripts.run_qi_ablation_study import run_qi_ablation_study

await run_qi_ablation_study(use_cached_explanations=True)

Total available rows: 6986
Non-noise log10(QI) range in dataset: -13.9999 to -12.0020
Loading existing training SCLKs from cda_dust_agent/data/ablation_study/train_sclks.json...
Total unique few-shot SCLKs selected across all 5 bins: 491
Generating reference explanations for 491 missing SCLKs...
Pre-rendering and caching few-shot spectrum images...
Saved updated explanations cache to qi_ablation_results/explanations_cache.json.
Successfully compiled explanations pool of size 491.
Pre-rendering target test spectrum images...
Consolidated request file created successfully with 2720 requests.
Uploading qi_ablation_results/requests_qi_ablation.jsonl to GCS bucket: turansgenaibb


Uploading to GCS: 100%|██████████| 18.4G/18.4G [35:59<00:00, 6.41MB/s]

Uploaded to gs://turansgenaibb/input/requests_qi_ablation.jsonl
Submitting Batch Job to Vertex AI...
Job Submitted! Name: projects/355771430623/locations/global/batchPredictionJobs/4615608526159478784
Polling job status every 30 seconds...
[10:12:19] Job State: JOB_STATE_PENDING


Uploading to GCS: 100%|██████████| 18.4G/18.4G [36:10<00:00, 6.41MB/s]

[10:12:51] Job State: JOB_STATE_PENDING
[10:13:22] Job State: JOB_STATE_PENDING
[10:13:53] Job State: JOB_STATE_PENDING
[10:14:24] Job State: JOB_STATE_PENDING
[10:14:56] Job State: JOB_STATE_PENDING
[10:15:27] Job State: JOB_STATE_PENDING
[10:15:59] Job State: JOB_STATE_PENDING
[10:16:29] Job State: JOB_STATE_PENDING
[10:17:01] Job State: JOB_STATE_PENDING
[10:17:32] Job State: JOB_STATE_PENDING
[10:18:04] Job State: JOB_STATE_PENDING
[10:18:35] Job State: JOB_STATE_PENDING
[10:19:06] Job State: JOB_STATE_PENDING
[10:19:38] Job State: JOB_STATE_PENDING
[10:20:09] Job State: JOB_STATE_PENDING
[10:20:40] Job State: JOB_STATE_PENDING
[10:21:12] Job State: JOB_STATE_PENDING
[10:21:43] Job State: JOB_STATE_PENDING
[10:22:14] Job State: JOB_STATE_PENDING
[10:22:46] Job State: JOB_STATE_PENDING
[10:23:17] Job State: JOB_STATE_PENDING
[10:23:48] Job State: JOB_STATE_PENDING
[10:24:19] Job State: JOB_STATE_PENDING
[10:24:51] Job State: JOB_STATE_PENDING
[10:25:22] Job State: JOB_STATE_PENDING


Uploading to GCS: 100%|██████████| 18.4G/18.4G [1:25:07<00:00, 3.61MB/s]

[11:01:23] Job State: JOB_STATE_SUCCEEDED
Job Completed Successfully!
Output directory in GCS: gs://turansgenaibb/output/prediction-model-2026-06-11T10:12:18.171489Z


Found 1 prediction JSONL files. Downloading and parsing...
Saved consolidated results to: qi_ablation_results/results_qi_ablation.parquet
Bin 0 ([-14.00, -13.60)): Mean Acc = 83.27% (std: 0.00%)
Bin 1 ([-13.60, -13.20)): Mean Acc = 84.01% (std: 0.00%)
Bin 2 ([-13.20, -12.80)): Mean Acc = 85.85% (std: 0.00%)
Bin 3 ([-12.80, -12.40)): Mean Acc = 80.51% (std: 0.00%)
Bin 4 ([-12.40, -12.00)): Mean Acc = 77.39% (std: 0.00%)
Saved statistics JSON to qi_ablation_results/bin_statistics.json
Saved final accuracy curve plot to qi_ablation_results/plots/qi_ablation_accuracy_curve.png


In [4]:
# Load and display accuracy statistics from the study
stats_path = "../qi_ablation_results/bin_statistics.json"
if os.path.exists(stats_path):
    with open(stats_path, 'r') as f:
        bin_stats = json.load(f)
    
    # Convert to pandas DataFrame for pretty representation
    rows = []
    for b_idx, stats in bin_stats.items():
        rows.append({
            "Bin": int(b_idx),
            "Log10(QI) Min": stats["log_qi_range"][0],
            "Log10(QI) Max": stats["log_qi_range"][1],
            "Mean Accuracy": f"{stats['mean_accuracy']:.2%}",
            "Std Dev": f"{stats['std_accuracy']:.2%}",
            "Run Accuracies": [f"{x:.1%}" for x in stats["run_accuracies"]]
        })
    df_stats = pd.DataFrame(rows)
    display(df_stats)
else:
    print("Statistics file not found yet. Run the cell above first.")

Statistics file not found yet. Run the cell above first.


### Visualizing the Study Results

In [5]:
# Display the consolidated accuracy curve
from IPython.display import Image, display

curve_path = "../qi_ablation_results/plots/qi_ablation_accuracy_curve.png"
if os.path.exists(curve_path):
    display(Image(filename=curve_path))
else:
    print("Accuracy curve plot not found.")

Accuracy curve plot not found.
